# Training-data QC: is this dataset fit to train on?

Checks every `.npy` tensor under a directory (recursing through the
lidar-point-count subfolders, same walk as `HMATensorDataset`) against the
known 4-channel layout:

| Channel | Name         | Meaning                                   |
|---------|--------------|--------------------------------------------|
| 0       | `dem_bic`    | 10m bicubic-upsampled DEM                 |
| 1       | `lidar_raw`  | Raw ATL08 photon elevation                |
| 2       | `mask`       | Binary photon-existence mask (0/1)        |
| 3       | `gt_dem`     | 10m ground-truth DEM                      |

What it checks, per file and in aggregate:
- **Structural**: loadable, correct ndim/channel count, dtype, NaN/Inf.
- **Mask sanity**: values are actually binary, per-file photon density,
  files with zero anchors at all.
- **Size adequacy**: big enough for `train_crop`/`val_crop` without
  excessive reflect-padding.
- **Plausibility**: elevation channels in a sane range (catches wrong
  units, no-data sentinels, flipped sign).
- **Cross-channel consistency**: `gt_dem` vs `dem_bic` residual (what the
  model has to learn), and `lidar_raw` vs `gt_dem` agreement at masked
  pixels (photon vs. optical-DEM agreement -- large disagreement usually
  means misalignment, not signal).
- **Duplicates**: exact-content hash collisions.
- **Global mean/std** of `dem_bic` and `gt_dem` elevation, computed in a
  single streaming pass -- paste into `GLOBAL_MEAN`/`GLOBAL_STD` in
  `dataloader_anchor.py` if this is a new mountain range (those constants
  are range-specific).

Edit `DATA_DIRS` in the next cell, then Run All. The last cell prints a
pass/fail-style summary you can act on directly.


In [ ]:
import os, glob, hashlib, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas as pd
    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False
    print('pandas not found -- per-file table will print as plain text instead.')

from tqdm.auto import tqdm


In [ ]:
def load_hma_npy(path):
    """Load one HMA tensor .npy file, tolerant of two container formats:

    Legacy: a plain (4, H, W) float array -- dem_bic, lidar_raw, mask, gt_dem.

    Extended: the same 4 image channels plus per-file georeferencing
    metadata appended after them --
        Ch5: GDAL geotransform, shape (6,) float64, (x0, dx, 0, y0, 0, -dy)
        Ch6: EPSG code, single int.
    These extra fields do not share the (H, W) shape of the image channels,
    so they can only be packed as a heterogeneous container (object-dtype
    array wrapping a list/tuple, or a dict) -- this auto-detects which.

    Returns (image, geotransform, epsg):
        image        -- (4, H, W) float32 ndarray
        geotransform -- (6,) float64 ndarray, or None
        epsg         -- int, or None
    """
    try:
        raw = np.load(path, allow_pickle=False)
    except ValueError:
        raw = np.load(path, allow_pickle=True)

    if raw.dtype != object and raw.ndim == 3 and raw.shape[0] == 4:
        return raw.astype(np.float32, copy=False), None, None

    obj = raw
    if isinstance(raw, np.ndarray) and raw.dtype == object:
        obj = raw.item() if raw.shape == () else raw

    if isinstance(obj, dict):
        def _get(*names):
            for n in names:
                if n in obj:
                    return obj[n]
            return None
        dem_bic      = _get("dem_bic", "Ch1", "ch1")
        lidar_raw    = _get("lidar_raw", "Ch2", "ch2")
        mask         = _get("mask", "Ch3", "ch3")
        gt_dem       = _get("gt_dem", "Ch4", "ch4")
        geotransform = _get("geotransform", "gt", "affine", "Ch5", "ch5")
        epsg         = _get("epsg", "EPSG", "Ch6", "ch6")
    elif isinstance(obj, (list, tuple, np.ndarray)) and len(obj) >= 4:
        dem_bic, lidar_raw, mask, gt_dem = obj[0], obj[1], obj[2], obj[3]
        geotransform = obj[4] if len(obj) > 4 else None
        epsg = obj[5] if len(obj) > 5 else None
    else:
        raise ValueError(
            "Unrecognized .npy container for " + str(path) + ": dtype=" + str(raw.dtype) +
            ", shape=" + str(getattr(raw, "shape", None)) + ", type=" + str(type(obj))
        )

    image = np.stack([
        np.asarray(dem_bic, dtype=np.float32),
        np.asarray(lidar_raw, dtype=np.float32),
        np.asarray(mask, dtype=np.float32),
        np.asarray(gt_dem, dtype=np.float32),
    ], axis=0)
    geotransform = np.asarray(geotransform, dtype=np.float64) if geotransform is not None else None
    epsg = int(epsg) if epsg is not None else None
    return image, geotransform, epsg


In [ ]:
# ============================================================================
# CONFIG -- edit for your dataset
# ============================================================================

# One or more directories to scan (each walked recursively for .npy files,
# same as HMATensorDataset). Point this at train, val, or both.
DATA_DIRS = [
    r'D:\Vaibhav\dataset_12m\tensors\train',
    r'D:\Vaibhav\dataset_12m\tensors\validation_contiguous',
]

EXPECTED_CHANNELS = 4
TRAIN_CROP = 128   # must fit without more than MAX_PAD_FRACTION reflect-padding
VAL_CROP   = 256

# Plausibility bounds for elevation channels (metres). Generous by default --
# tighten if you know the true range for this mountain range.
ELEV_MIN_PLAUSIBLE = -500.0
ELEV_MAX_PLAUSIBLE = 9000.0

# Flag thresholds
MAX_PAD_FRACTION            = 0.5   # file needs >50% padding on an axis -> flag as too small
PHOTON_DISAGREEMENT_FLAG_M  = 50.0  # |lidar_raw - gt_dem| at masked px above this -> flag file
MASK_TOLERANCE               = 1e-6  # values outside {0,1} +/- this -> flag as non-binary

N_SAMPLE_PIXELS_PER_FILE = 400   # reservoir sample size per file, for histograms only
RANDOM_SEED = 0
CHECK_DUPLICATES = True          # exact-content hash check (cheap, data's already in RAM)

np.random.seed(RANDOM_SEED)

filepaths = []
for d in DATA_DIRS:
    if not Path(d).exists():
        print(f'WARNING: path does not exist -- {d}')
        continue
    for root, _, files in os.walk(d):
        for f in files:
            if f.endswith('.npy'):
                filepaths.append(os.path.join(root, f))

print(f'Found {len(filepaths)} .npy files across {len(DATA_DIRS)} director{"y" if len(DATA_DIRS)==1 else "ies"}.')


In [ ]:
# ============================================================================
# SINGLE-PASS SCAN
# ============================================================================
# Streaming accumulators (float64) for exact global mean/std without holding
# every pixel in memory.
sum_dem = sumsq_dem = n_dem = 0.0
sum_gt  = sumsq_gt  = n_gt  = 0.0

rows = []                  # one dict per successfully-loaded file
load_errors = []           # (path, exception str)
shape_errors = []          # paths with wrong ndim/channel count
nan_files = []             # paths containing NaN/Inf
non_binary_mask_files = []
zero_mask_files = []
undersized_files = []      # need > MAX_PAD_FRACTION padding on an axis
implausible_elev_files = []
high_photon_disagreement_files = []
content_hashes = defaultdict(list)   # hash -> [paths]  (duplicate detection)

sample_dem_vals = []
sample_residual_vals = []   # gt_dem - dem_bic (all pixels)
sample_delta_vals = []      # lidar_raw - dem_bic at masked pixels
sample_photon_disagree_vals = []  # |lidar_raw - gt_dem| at masked pixels
per_file_mask_density = []
n_with_geotransform = 0

for path in tqdm(filepaths, desc='Scanning files'):
    try:
        data, geotransform, epsg = load_hma_npy(path)
    except Exception as e:
        load_errors.append((path, str(e)))
        continue

    if geotransform is not None:
        n_with_geotransform += 1

    if data.ndim != 3 or data.shape[0] != EXPECTED_CHANNELS:
        shape_errors.append((path, data.shape))
        continue

    c, h, w = data.shape
    dem_bic, lidar_raw, mask, gt_dem = data[0], data[1], data[2], data[3]

    # ---- NaN / Inf -----------------------------------------------------
    finite_mask_all = np.isfinite(data)
    n_nonfinite = int((~finite_mask_all).sum())
    if n_nonfinite > 0:
        nan_files.append((path, n_nonfinite))

    dem_finite = np.isfinite(dem_bic)
    gt_finite  = np.isfinite(gt_dem)

    # ---- Mask sanity -----------------------------------------------------
    mask_vals = mask[np.isfinite(mask)]
    is_binary = np.all(
        (np.abs(mask_vals) < MASK_TOLERANCE) | (np.abs(mask_vals - 1.0) < MASK_TOLERANCE)
    ) if mask_vals.size > 0 else True
    if not is_binary:
        non_binary_mask_files.append(path)

    mask_bin = (mask > 0.5) & np.isfinite(mask)
    n_photons = int(mask_bin.sum())
    density = n_photons / (h * w)
    per_file_mask_density.append(density)
    if n_photons == 0:
        zero_mask_files.append(path)

    # ---- Size adequacy ---------------------------------------------------
    pad_h_frac = max(0, TRAIN_CROP - h) / TRAIN_CROP
    pad_w_frac = max(0, TRAIN_CROP - w) / TRAIN_CROP
    if pad_h_frac > MAX_PAD_FRACTION or pad_w_frac > MAX_PAD_FRACTION:
        undersized_files.append((path, (h, w)))

    # ---- Elevation plausibility -------------------------------------------
    dem_valid = dem_bic[dem_finite]
    gt_valid  = gt_dem[gt_finite]
    implausible = False
    if dem_valid.size > 0 and (dem_valid.min() < ELEV_MIN_PLAUSIBLE or dem_valid.max() > ELEV_MAX_PLAUSIBLE):
        implausible = True
    if gt_valid.size > 0 and (gt_valid.min() < ELEV_MIN_PLAUSIBLE or gt_valid.max() > ELEV_MAX_PLAUSIBLE):
        implausible = True
    if implausible:
        implausible_elev_files.append(path)

    # ---- Streaming global mean/std ----------------------------------------
    if dem_valid.size > 0:
        sum_dem   += float(dem_valid.sum(dtype=np.float64))
        sumsq_dem += float(np.square(dem_valid, dtype=np.float64).sum())
        n_dem     += dem_valid.size
    if gt_valid.size > 0:
        sum_gt   += float(gt_valid.sum(dtype=np.float64))
        sumsq_gt += float(np.square(gt_valid, dtype=np.float64).sum())
        n_gt     += gt_valid.size

    # ---- Cross-channel consistency -----------------------------------------
    both_finite = dem_finite & gt_finite
    residual = (gt_dem - dem_bic)[both_finite]
    residual_mean = float(residual.mean()) if residual.size > 0 else float('nan')
    residual_std  = float(residual.std())  if residual.size > 0 else float('nan')

    if n_photons > 0:
        delta_at_mask = (lidar_raw - dem_bic)[mask_bin & dem_finite & np.isfinite(lidar_raw)]
        photon_disagree = np.abs(lidar_raw - gt_dem)[mask_bin & gt_finite & np.isfinite(lidar_raw)]
        delta_mean = float(delta_at_mask.mean()) if delta_at_mask.size > 0 else float('nan')
        delta_std  = float(delta_at_mask.std())  if delta_at_mask.size > 0 else float('nan')
        disagree_mean = float(photon_disagree.mean()) if photon_disagree.size > 0 else float('nan')
        disagree_max  = float(photon_disagree.max())  if photon_disagree.size > 0 else float('nan')
        if disagree_mean > PHOTON_DISAGREEMENT_FLAG_M:
            high_photon_disagreement_files.append((path, disagree_mean))
    else:
        delta_mean = delta_std = disagree_mean = disagree_max = float('nan')

    # ---- Duplicate detection (content hash) --------------------------------
    if CHECK_DUPLICATES:
        h_hash = hashlib.md5(data.tobytes()).hexdigest()
        content_hashes[h_hash].append(path)

    # ---- Reservoir sampling for histograms ----------------------------------
    if dem_valid.size > 0:
        idx = np.random.choice(dem_valid.size, size=min(N_SAMPLE_PIXELS_PER_FILE, dem_valid.size), replace=False)
        sample_dem_vals.append(dem_valid[idx])
    if residual.size > 0:
        idx = np.random.choice(residual.size, size=min(N_SAMPLE_PIXELS_PER_FILE, residual.size), replace=False)
        sample_residual_vals.append(residual[idx])
    if n_photons > 0 and delta_at_mask.size > 0:
        sample_delta_vals.append(delta_at_mask)
        sample_photon_disagree_vals.append(photon_disagree)

    rows.append({
        'path': path,
        'height': h, 'width': w,
        'dem_min': float(dem_valid.min()) if dem_valid.size else float('nan'),
        'dem_max': float(dem_valid.max()) if dem_valid.size else float('nan'),
        'dem_mean': float(dem_valid.mean()) if dem_valid.size else float('nan'),
        'gt_mean': float(gt_valid.mean()) if gt_valid.size else float('nan'),
        'mask_density': density,
        'n_photons': n_photons,
        'residual_mean': residual_mean,
        'residual_std': residual_std,
        'delta_mean_at_mask': delta_mean,
        'delta_std_at_mask': delta_std,
        'photon_gt_disagree_mean': disagree_mean,
        'photon_gt_disagree_max': disagree_max,
        'n_nonfinite': n_nonfinite,
    })

print(f'Scan complete. {len(rows)} files usable, {len(load_errors)} failed to load, '
      f'{len(shape_errors)} had wrong shape.')
print(f'{n_with_geotransform} files carried the extended geotransform/EPSG metadata '
      f'(loaded fine either way -- load_hma_npy() strips it to the 4-channel image).')


In [ ]:
# ============================================================================
# PER-FILE TABLE + GLOBAL MEAN/STD
# ============================================================================
if HAVE_PANDAS:
    df = pd.DataFrame(rows)
    display(df.describe(include='all'))
else:
    df = None
    print(f'{len(rows)} rows collected (install pandas for a nicer summary table).')

GLOBAL_MEAN_DEM = sum_dem / n_dem if n_dem > 0 else float('nan')
GLOBAL_STD_DEM  = math.sqrt(max(sumsq_dem / n_dem - GLOBAL_MEAN_DEM**2, 0.0)) if n_dem > 0 else float('nan')
GLOBAL_MEAN_GT  = sum_gt / n_gt if n_gt > 0 else float('nan')
GLOBAL_STD_GT   = math.sqrt(max(sumsq_gt / n_gt - GLOBAL_MEAN_GT**2, 0.0)) if n_gt > 0 else float('nan')

print('\n--- Global elevation statistics (exact, single-pass over all valid pixels) ---')
print(f'dem_bic : mean={GLOBAL_MEAN_DEM:.4f}  std={GLOBAL_STD_DEM:.4f}  (n={n_dem:,})')
print(f'gt_dem  : mean={GLOBAL_MEAN_GT:.4f}  std={GLOBAL_STD_GT:.4f}  (n={n_gt:,})')
print('\nIf this is a new mountain range, paste into dataloader_anchor.py:')
print(f'GLOBAL_MEAN = {GLOBAL_MEAN_DEM:.4f}')
print(f'GLOBAL_STD = {GLOBAL_STD_DEM:.4f}')


In [ ]:
# ============================================================================
# DISTRIBUTIONS
# ============================================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

if sample_dem_vals:
    axes[0, 0].hist(np.concatenate(sample_dem_vals), bins=60, color='steelblue')
axes[0, 0].set_title('dem_bic elevation (sampled pixels)')
axes[0, 0].set_xlabel('metres')

if sample_residual_vals:
    axes[0, 1].hist(np.concatenate(sample_residual_vals), bins=60, color='darkorange')
axes[0, 1].set_title('gt_dem - dem_bic residual\n(what the model has to learn)')
axes[0, 1].set_xlabel('metres')

axes[0, 2].hist(per_file_mask_density, bins=60, color='seagreen')
axes[0, 2].set_title('Photon mask density per file')
axes[0, 2].set_xlabel('fraction of pixels with a photon')

if sample_delta_vals:
    axes[1, 0].hist(np.concatenate(sample_delta_vals), bins=60, color='indianred')
axes[1, 0].set_title('lidar_raw - dem_bic at masked pixels\n(anchor delta)')
axes[1, 0].set_xlabel('metres')

if sample_photon_disagree_vals:
    axes[1, 1].hist(np.concatenate(sample_photon_disagree_vals), bins=60, color='purple')
axes[1, 1].set_title('|lidar_raw - gt_dem| at masked pixels\n(photon vs. optical-DEM agreement)')
axes[1, 1].set_xlabel('metres')

if HAVE_PANDAS and len(df) > 0:
    axes[1, 2].scatter(df['dem_mean'], df['residual_std'], s=8, alpha=0.5)
axes[1, 2].set_title('Per-file dem_mean vs. residual_std\n(spot elevation-dependent quality issues)')
axes[1, 2].set_xlabel('dem_bic mean (m)')
axes[1, 2].set_ylabel('residual std (m)')

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# VISUAL SPOT-CHECK: a handful of random tiles, all 4 channels
# ============================================================================
N_SPOT_CHECK = 4
spot_paths = np.random.choice([r['path'] for r in rows], size=min(N_SPOT_CHECK, len(rows)), replace=False)

fig, axes = plt.subplots(len(spot_paths), 4, figsize=(16, 4 * len(spot_paths)))
if len(spot_paths) == 1:
    axes = axes[np.newaxis, :]

for i, p in enumerate(spot_paths):
    data = np.load(p).astype(np.float32)
    dem_bic, lidar_raw, mask, gt_dem = data[0], data[1], data[2], data[3]

    vmin, vmax = np.nanmin(gt_dem), np.nanmax(gt_dem)
    im0 = axes[i, 0].imshow(dem_bic, cmap='terrain', vmin=vmin, vmax=vmax)
    axes[i, 0].set_title(f'dem_bic\n{Path(p).name}', fontsize=8)
    im1 = axes[i, 1].imshow(gt_dem, cmap='terrain', vmin=vmin, vmax=vmax)
    axes[i, 1].set_title('gt_dem', fontsize=8)
    axes[i, 2].imshow(mask, cmap='gray')
    axes[i, 2].set_title(f'mask ({int(mask.sum())} photons)', fontsize=8)
    im3 = axes[i, 3].imshow(gt_dem - dem_bic, cmap='RdBu_r',
                             vmin=-np.nanmax(np.abs(gt_dem - dem_bic)),
                             vmax=np.nanmax(np.abs(gt_dem - dem_bic)))
    axes[i, 3].set_title('gt_dem - dem_bic', fontsize=8)
    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# FINAL REPORT
# ============================================================================
def _list(name, items, limit=10):
    print(f'  {name}: {len(items)}')
    for x in items[:limit]:
        print(f'    - {x}')
    if len(items) > limit:
        print(f'    ... and {len(items) - limit} more')

duplicate_groups = {h: paths for h, paths in content_hashes.items() if len(paths) > 1} if CHECK_DUPLICATES else {}

print('=' * 70)
print('TRAINING DATA QC REPORT')
print('=' * 70)
print(f'Files found:        {len(filepaths)}')
print(f'Files usable:        {len(rows)}')
print()

ok = True

if load_errors:
    ok = False
    print('FAILED TO LOAD:')
    _list('files', [p for p, _ in load_errors])
if shape_errors:
    ok = False
    print('WRONG SHAPE (expected 3D, 4 channels):')
    _list('files', [f'{p} -> shape {s}' for p, s in shape_errors])
if nan_files:
    ok = False
    print('CONTAIN NaN/Inf:')
    _list('files', [f'{p} ({n} bad values)' for p, n in nan_files])
if non_binary_mask_files:
    ok = False
    print('MASK CHANNEL NOT BINARY (0/1):')
    _list('files', non_binary_mask_files)
if implausible_elev_files:
    ok = False
    print(f'ELEVATION OUTSIDE PLAUSIBLE RANGE [{ELEV_MIN_PLAUSIBLE}, {ELEV_MAX_PLAUSIBLE}] m:')
    _list('files', implausible_elev_files)
if undersized_files:
    print(f'UNDERSIZED (need >{MAX_PAD_FRACTION*100:.0f}% reflect-padding for train_crop={TRAIN_CROP}) -- usable but low quality:')
    _list('files', [f'{p} -> {s}' for p, s in undersized_files])
if high_photon_disagreement_files:
    print(f'HIGH PHOTON/GT DISAGREEMENT (mean > {PHOTON_DISAGREEMENT_FLAG_M}m) -- check alignment/projection:')
    _list('files', [f'{p} (mean {d:.1f}m)' for p, d in high_photon_disagreement_files])
if zero_mask_files:
    print('ZERO PHOTON COVERAGE (contribute no pin/aux loss signal):')
    _list('files', zero_mask_files)
if duplicate_groups:
    print('DUPLICATE CONTENT (exact hash match):')
    for h, paths in list(duplicate_groups.items())[:10]:
        print(f'    - {paths}')

if ok and not undersized_files and not high_photon_disagreement_files and not zero_mask_files and not duplicate_groups:
    print('No issues found. Dataset looks structurally sound.')

print()
print('-' * 70)
print(f'Mask density   : mean={np.mean(per_file_mask_density)*100:.3f}%  '
      f'median={np.median(per_file_mask_density)*100:.3f}%  '
      f'files with 0 photons={len(zero_mask_files)}/{len(rows)}')
print(f'Global dem_bic : mean={GLOBAL_MEAN_DEM:.2f}m  std={GLOBAL_STD_DEM:.2f}m')
print(f'Global gt_dem  : mean={GLOBAL_MEAN_GT:.2f}m  std={GLOBAL_STD_GT:.2f}m')
print('-' * 70)
print('HARD BLOCKERS (fix before training)      : load errors, shape errors, NaN/Inf, non-binary mask, implausible elevation')
print('SOFT WARNINGS (train works, but check)    : undersized tiles, high photon/GT disagreement, zero-mask tiles, duplicates')
